# First baseline experiment

This notebook prepares a majority-class reference and a logistic-regression model.
You control installation, preparation, and training separately. All three switches default to false.
Exporting this notebook as a report does not start training with these defaults.
Use a CPU runtime. These models do not require a GPU.


In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

RUN_SETUP = False
RUN_PREPARATION = False
RUN_TRAINING = False

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/baseline-workflow"
local_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/baseline.yaml").exists()), None
)
PROJECT_DIR = local_root or Path("/content/sleep-classifier")

## 1. Open the checkout and install uv

For a fresh Colab runtime, enable RUN_SETUP first.
The installer uses Astral's official installation script. Existing checkouts remain unchanged.
Select the experiment branch before its merge. Use main after the experiment branch merges.


In [ ]:
if RUN_SETUP:
    if not PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, REPOSITORY_URL, str(PROJECT_DIR)], check=True
        )
    if shutil.which("uv") is None:
        installer = subprocess.run(
            ["curl", "-LsSf", "https://astral.sh/uv/install.sh"],
            check=True,
            capture_output=True,
            text=True,
        )
        subprocess.run(
            ["sh", "-s", "--", "--no-modify-path"], input=installer.stdout, text=True, check=True
        )
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    subprocess.run(
        ["uv", "sync", "--locked", "--group", "baseline", "--group", "notebook"],
        cwd=PROJECT_DIR,
        check=True,
    )
    subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, check=True)
else:
    print("Setup skipped")

## 2. Prepare data

Enable RUN_PREPARATION on a fresh checkout. The archive requires about 550 MB of network transfer.
Integrity checks include all participants. Feature preparation opens training and validation observations only.
Preparation uses the saved split and coverage rule. It does not train a model.


In [ ]:
if RUN_PREPARATION:
    commands = [
        [
            "sleep_classifier.download_cohort",
            "--archive",
            "data/sleep-accel-1.0.0.zip",
            "--destination",
            "data/raw/sleep-accel/1.0.0",
        ],
        ["sleep_classifier.cohort", "inventory", "--config", "configs/cohort.yaml"],
        ["sleep_classifier.cohort", "split", "--config", "configs/cohort.yaml"],
        ["sleep_classifier.prepare", "--config", "configs/prepare.yaml"],
    ]
    for arguments in commands:
        subprocess.run(
            ["uv", "run", "--locked", "--group", "baseline", "python", "-m", *arguments],
            cwd=PROJECT_DIR,
            check=True,
        )
else:
    print("Data preparation skipped")

## 3. Train when ready

Enable RUN_TRAINING to fit both baselines. Review configs/baseline.yaml before running this cell.
The scaler fits eligible training rows only. Validation rows receive the fitted transformation.
The classifiers use the six explicit sensor features. Labels and coverage fields remain excluded from model inputs.

Equivalent command from the checkout:
```bash
uv run --locked --group baseline python -m sleep_classifier.baseline --config configs/baseline.yaml --train
```


In [ ]:
if RUN_TRAINING:
    subprocess.run(
        [
            "uv",
            "run",
            "--locked",
            "--group",
            "baseline",
            "python",
            "-m",
            "sleep_classifier.baseline",
            "--config",
            "configs/baseline.yaml",
            "--train",
        ],
        cwd=PROJECT_DIR,
        check=True,
    )
else:
    print("Training skipped. Enable RUN_TRAINING only when you choose to fit the models.")

## 4. Review saved validation results

Metrics describe retained validation epochs. Coverage uses all scored validation epochs as its denominator.
Compare accuracy, macro-F1, REM precision/recall/F1, and individual participants.
The final test set remains reserved for the fixed model protocol.
Old metrics remain available until you replace them with a new training run.
Inspect the saved configuration and provenance before comparing runs.


In [ ]:
metrics_path = PROJECT_DIR / "artifacts/baseline/metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("Saved configuration:", metrics["config"])
    for name, model in metrics["models"].items():
        print(name)
        print("Accuracy:", model["accuracy"], "Macro-F1:", model["macro_f1"])
        print("Coverage:", model["coverage"])
        print("REM:", model["classes"]["3"])
        print("Confusion matrix (wake, light, deep, REM):", model["confusion_matrix"])
else:
    print("No saved baseline results. Run training when ready.")

## 5. Export the report

Run this command after training. The report runner skips the tagged training cell, including when RUN_TRAINING is enabled interactively.
The report reads the saved metrics and writes HTML only.

```bash
uv run --locked --group notebook --group baseline python scripts/execute_notebook.py --notebook notebooks/05_baseline_experiment.ipynb
```

Keep the source notebook switches false before committing or exporting it.
The HTML report is written to artifacts/notebooks/05_baseline_experiment.executed.html.
